# Deterministic value iteration

Upload this file to GitHub to view it. To run it, open [Google Colab](https://colab.research.google.com/), select **GitHub** in the open dialog, and paste the notebook link (or upload the file). Run the cells from top to bottom. A CPU is sufficient; the first code cell installs the dependencies. This notebook is self-contained and does not require the `.py` files. You can also use it in Jupyter.

4×4 Gridworld: start at (2, 1), treasure at (4, 4) with reward +1, and fire at (4, 3) with reward −1. Coordinates start at the bottom-left corner. Actions are deterministic. Hitting a wall leaves the agent in the same cell. Rewards are received on entry; terminal state values remain zero.

In [ ]:
%pip install -q matplotlib
%matplotlib inline

## Parameters

Modify `GAMMA`, `ITERATIONS`, and `PLOT_EVERY`, then rerun the following cells.

In [ ]:
SIZE = 4
GAMMA = 0.9  # Discount future rewards: reaching the treasure sooner is better.
ITERATIONS = 10
PLOT_EVERY = 1
START = (2, 1)
TERMINALS = {(4, 4): 1.0, (4, 3): -1.0}
ACTIONS = [(0, 1), (0, -1), (-1, 0), (1, 0)]
STATES = [(x, y) for y in range(1, SIZE + 1) for x in range(1, SIZE + 1)]

## Model and Bellman update

Each update uses the values from the previous iteration.

In [ ]:
def transition(state, action):
    """Return the next state and reward for a deterministic action."""
    if state in TERMINALS:
        return state, 0.0  # The episode has ended; no further rewards.
    x, y = state
    dx, dy = action
    next_state = (min(SIZE, max(1, x + dx)), min(SIZE, max(1, y + dy)))
    reward = TERMINALS.get(next_state, 0.0)
    return next_state, reward


def bellman_update(values):
    """Perform one synchronous sweep of the Bellman optimality equation."""
    new_values = {}
    for state in STATES:
        if state in TERMINALS:
            # V(terminal) = 0: its entry reward has already been received.
            new_values[state] = 0.0
            continue
        action_values = []
        for action in ACTIONS:
            next_state, reward = transition(state, action)
            # Q(s,a) = R(s,a,s') + gamma * V(s'). No sampling is needed.
            action_values.append(reward + GAMMA * values[next_state])
        new_values[state] = max(action_values)
    # Every state uses the PREVIOUS sweep, independent of iteration order.
    return new_values

## Visualization

In [ ]:
def plot_values(ax, values, iteration):
    """Draw a table with coordinates matching the reference picture."""
    ax.clear()
    for x, y in STATES:
        state = (x, y)
        color = "white"
        label = f"V = {values[state]:.3f}"
        if state in TERMINALS:
            reward = TERMINALS[state]
            color = "#d8f3dc" if reward > 0 else "#ffdad6"
            label += f"\nR = {reward:+.0f} (terminal)"
        elif state == START:
            color = "#dceeff"
            label += "\nS (start)"
        # Matplotlib is only needed for plotting, not for the algorithm.
        from matplotlib.patches import Rectangle
        ax.add_patch(Rectangle((x - 0.5, y - 0.5), 1, 1,
                               facecolor=color, edgecolor="black"))
        ax.text(x, y, label, ha="center", va="center", fontsize=10)
    ax.set(xlim=(0.5, SIZE + 0.5), ylim=(0.5, SIZE + 0.5),
           xticks=range(1, SIZE + 1), yticks=range(1, SIZE + 1),
           xlabel="x", ylabel="y",
           title=f"Value iteration — sweep {iteration} (gamma = {GAMMA})")
    ax.set_aspect("equal")

## Run value iteration

Display the initial state and selected iterations directly in the notebook, without external windows.

In [ ]:
import matplotlib.pyplot as plt

assert ITERATIONS >= 1 and PLOT_EVERY >= 1
values = {state: 0.0 for state in STATES}
value_history = [values.copy()]
deltas = []

def show_values(values, iteration):
    fig, ax = plt.subplots(figsize=(7, 7))
    plot_values(ax, values, iteration)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

show_values(values, 0)
for iteration in range(1, ITERATIONS + 1):
    new_values = bellman_update(values)
    delta = max(abs(new_values[s] - values[s]) for s in STATES)
    values = new_values
    value_history.append(values.copy())
    deltas.append(delta)
    if iteration % PLOT_EVERY == 0 or iteration == ITERATIONS:
        print(f"Sweep {iteration:3d}: max value change = {delta:.6f}")
        show_values(values, iteration)

## Convergence

If the final change is still large, increase `ITERATIONS`.

In [ ]:
fig, ax = plt.subplots()
ax.plot(range(1, ITERATIONS + 1), deltas, marker="o")
ax.set(xlabel="Iteration", ylabel="Maximum value change", title="Convergence")
ax.grid(alpha=0.3)
plt.show()
plt.close(fig)